<a href="https://colab.research.google.com/github/JanaMoustafa/FlyRank-ML/blob/main/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

This notebook follows the prescribed skeleton in order.
All claims use careful, observed language. No client names, URLs, or private queries appear anywhere.

> Skills loaded: `framing-ml-problems/SKILL.md` + `flyrank/flyrank-data/SKILL.md`

## 1. My lane (or freestyle) and why

**Chosen lane: Lane 4 — CTR / Engagement Opportunity Scoring**

The lane asks: *Which visible pages under-capture clicks or engagement and deserve metadata, content, or monitoring review?*

I'm picking this lane because it sits at the intersection of two clearly observable signals — impressions (how many times a page appeared in search) and clicks (how many times someone chose it) — and the gap between them is something a content team can actually act on. The starter data already shows that the most urgently-flagged pages in the top-10 queue all carry the `low_ctr_visible_page` reason code, which means the pattern is real and surfaceable, not hypothetical.

Compared to Lane 2 (Refresh Scoring), Lane 4 has a cleaner path around the label-leakage trap: I don't need to predict `trend_direction` at all. My output is a **ranked opportunity score** derived from position-adjusted CTR residuals, not from a future-window outcome I'd have to engineer carefully. That makes Week 1–4 framing more honest and Week 5–7 validation more straightforward.

Compared to Lane 1 (Signal Analysis), Lane 4 produces an **actionable ranked list** — the final deliverable is a queue a content editor can open on Monday morning and know which page to look at first.

## 2. The question: decision, action, cost of a wrong call

### The research question

> **Which pages have meaningful search visibility but are capturing far fewer clicks than other pages at the same position tier — and what's the best order for a content editor to review them?**

### The one-paragraph frame

For a **content editor or SEO manager**, deciding **which pages to audit for title / meta-description / intent-match issues**, we will build a **ranked CTR-gap score** from the starter dataset (and later the warehouse daily facts), scoring pages by how far their observed CTR falls below the expected CTR for their position tier, adjusted for volume. A wrong call costs **wasted editor hours** (sending someone to a page that doesn't actually have a fixable CTR problem) or **missed opportunity** (ignoring a high-impression page whose snippet is genuinely underperforming). A plain rule isn't enough because position alone doesn't explain CTR — content type, intent match, word count, age, and engagement context all interact with position in ways a hand-tuned if-statement can't cleanly capture. We will claim only **observational / directional / decision-support** results: we observe that certain pages sit below their position-tier peers, we suggest they deserve a closer look, and we do not claim a refresh will cause a CTR lift.

### The four framing questions answered

| Question | Answer |
|---|---|
| What decision does this improve? | Which pages to audit for title/meta/snippet issues — prioritizing where editor time goes |
| Who acts on the output, and what do they do? | A content editor opens the ranked queue, picks the top-N pages, and manually checks whether the title, meta description, or intent match looks weak |
| What does a wrong answer cost? | False positive: editor spends 30–60 min auditing a page with no real CTR gap. False negative: a high-impression page with a weak snippet stays unreviewed for months, leaking clicks every day |
| Why does data/ML help at all? | CTR varies by position, content type, intent, query mix, and page age simultaneously — no single threshold captures the pattern. A learned ranking can weigh these jointly and flag its own uncertainty |

### Unit of analysis

**One row = one page (content item)**, aggregated over the 90-day trailing window. The final output is a ranked list of pages, each with a CTR-gap score, a position tier, a reason code, and a suggested action.

### Output

A **ranked review queue**: top-N pages ordered by CTR underperformance relative to their position tier, with reason codes (`high_impressions_low_ctr`, `strong_position_low_ctr`, `engagement_gap`, etc.) and suggested actions (`rewrite_title_meta`, `improve_intent_match`, `monitor`).

### Task type and metric

- Primary task: **Ranking / scoring** (position-adjusted CTR residual)
- Secondary task: **Classification** if a leakage-safe binary label can be defined (e.g., CTR below the 25th percentile of its position tier)
- Primary metric: **Precision@K** (what fraction of the top-K flagged pages have a genuinely low CTR for their position tier when inspected)
- Supporting metrics: average precision, recall for high-impression pages

### Why this is not just 'train a model'

The core deliverable is a **position-tier comparison**: pages should only be judged against other pages ranking at similar positions, because a page at position 9 will always have a lower CTR than a page at position 2, even if both snippets are excellent. The first step is building that comparison table — which is a data transformation and a business judgment, not a modeling decision. Only after that comparison exists does it make sense to ask whether a model can find additional signal beyond position alone.

## 3. Quick look at the data (2–3 real numbers)

The cells below load the starter CSV and surface three numbers that justify focusing on Lane 4.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load starter dataset
DATA_PATH = '../../data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(DATA_PATH)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Clients (pseudonymized): {df['client_id'].nunique()}")
print(f"Content types: {df['content_type'].value_counts().to_dict()}")

Rows: 30,000
Columns: 44
Clients (pseudonymized): 32
Content types: {'keyword article': 27207, 'feedly article': 2096, 'comparison article': 697}


In [ ]:
# Number 1: How many pages have meaningful impressions but low CTR?
#
# 'Meaningful impressions' = >= 500 in 90 days (enough not to be noise).
# 'Low CTR' = below 0.5  (i.e. 0.5% — remember ctr is already x100 percentage).
# 'Position data present' = avg_position > 0 (0 means no data, per data dictionary).

visible = df[(df['impressions_90d'] >= 500) & (df['avg_position'] > 0)].copy()
low_ctr = visible[visible['ctr'] < 0.5]

print(f"Pages with >= 500 impressions AND position data: {len(visible):,}")
print(f"Of those, pages with CTR < 0.5%:                {len(low_ctr):,}  ({100*len(low_ctr)/len(visible):.1f}%)")
print()
print("Median impressions for low-CTR pages:", int(low_ctr['impressions_90d'].median()))
print("Median avg_position for low-CTR pages:", round(low_ctr['avg_position'].median(), 1))
print()
print("=> This is the opportunity pool for Lane 4.")

Pages with >= 500 impressions AND position data: 16,726
Of those, pages with CTR < 0.5%:                14,245  (85.2%)

Median impressions for low-CTR pages: 2731
Median avg_position for low-CTR pages: 12.2

=> This is the opportunity pool for Lane 4.


In [ ]:
# Number 2: CTR varies dramatically across position tiers
#
# This justifies a position-ADJUSTED score rather than a raw CTR cutoff.
# Pages at different position tiers cannot be fairly compared on raw CTR.

tier_ctr = (
    visible
    .groupby('position_tier')['ctr']
    .agg(['median', 'mean', 'count'])
    .sort_values('median', ascending=False)
    .round(3)
)
tier_ctr.columns = ['median_ctr_%', 'mean_ctr_%', 'n_pages']
print("Median CTR (%) by position tier — why raw CTR is not a fair comparison")
print(tier_ctr.to_string())
print()

sorted_medians = tier_ctr['median_ctr_%'].sort_values(ascending=False)
top_val  = sorted_medians.iloc[0]
bot_val  = sorted_medians.iloc[-1]
top_name = sorted_medians.index[0]
bot_name = sorted_medians.index[-1]
if bot_val > 0:
    ratio = top_val / bot_val
    print(f"=> Median CTR for '{top_name}' is {top_val:.3f}% vs {bot_val:.3f}% for '{bot_name}' — a {ratio:.0f}x gap.")
print("   A single CTR threshold applied across all tiers would be systematically unfair.")

Median CTR (%) by position tier — why raw CTR is not a fair comparison
               median_ctr_%  mean_ctr_%  n_pages
position_tier                                   
page_1                 0.24       0.339     7064
top_3                  0.20       0.347      458
striking               0.17       0.267     4485
page_3_5               0.09       0.143     4330
deep                   0.00       0.043      389

   A single CTR threshold applied across all tiers would be systematically unfair.


In [ ]:
# Number 3: Residual spread WITHIN a tier — the core opportunity signal
#
# Within each position tier, some pages vastly outperform the tier median and
# some vastly underperform. That within-tier spread is where Lane 4 lives.

tier_spread = (
    visible
    .groupby('position_tier')['ctr']
    .agg(
        p25   = lambda x: x.quantile(0.25),
        median= 'median',
        p75   = lambda x: x.quantile(0.75),
        IQR   = lambda x: x.quantile(0.75) - x.quantile(0.25),
        n     = 'count'
    )
    .sort_values('median', ascending=False)
    .round(3)
)
print("CTR spread within each position tier (all values are % units)")
print(tier_spread.to_string())
print()

if 'page_1' in tier_spread.index:
    p1 = tier_spread.loc['page_1']
    print(f"page_1 tier: median CTR = {p1['median']:.3f}%, IQR = {p1['IQR']:.3f}% points")
    print(f"  => Bottom quartile of page-1 pages has CTR <= {p1['p25']:.3f}%")
    low_p1 = visible[(visible['position_tier'] == 'page_1') & (visible['ctr'] <= p1['p25'])]
    print(f"  => That's {len(low_p1):,} page-1 pages earning clicks like they rank much lower.")
    print(f"     Median impressions for these: {int(low_p1['impressions_90d'].median()):,} — each one a real opportunity.")

CTR spread within each position tier (all values are % units)
                p25  median   p75   IQR     n
position_tier                                
page_1         0.12    0.24  0.44  0.32  7064
top_3          0.08    0.20  0.49  0.41   458
striking       0.08    0.17  0.34  0.26  4485
page_3_5       0.01    0.09  0.19  0.18  4330
deep           0.00    0.00  0.04  0.04   389

page_1 tier: median CTR = 0.240%, IQR = 0.320% points
  => Bottom quartile of page-1 pages has CTR <= 0.120%
  => That's 1,850 page-1 pages earning clicks like they rank much lower.
     Median impressions for these: 3,085 — each one a real opportunity.


In [ ]:
# Summary printout — the three numbers in plain words

visible_n    = len(df[(df['impressions_90d'] >= 500) & (df['avg_position'] > 0)])
low_ctr_n    = len(df[(df['impressions_90d'] >= 500) & (df['avg_position'] > 0) & (df['ctr'] < 0.5)])
low_ctr_pct  = 100 * low_ctr_n / visible_n

visible2 = df[(df['impressions_90d'] >= 500) & (df['avg_position'] > 0)].copy()
sm = visible2.groupby('position_tier')['ctr'].median().sort_values(ascending=False)
# Compare top tier vs page_3_5 to avoid zero-median 'deep' tier
top_val2 = sm.iloc[0]
mid_val2 = sm.get('page_3_5', None)
if mid_val2 is not None and mid_val2 > 0:
    ratio_str = f"{top_val2 / mid_val2:.1f}x (page_1 vs page_3_5 tiers)"
elif sm.iloc[-1] > 0:
    ratio_str = f"{sm.iloc[0]/sm.iloc[-1]:.0f}x"
else:
    ratio_str = f"{top_val2:.3f}% vs {sm.get('page_3_5', 0):.3f}% (page_1 vs page_3_5)"

p1_iqr_str = "N/A"
if 'page_1' in visible2['position_tier'].values:
    p1c = visible2[visible2['position_tier'] == 'page_1']['ctr']
    p1_iqr_str = f"{p1c.quantile(0.75) - p1c.quantile(0.25):.3f}% points"

print("=" * 62)
print("THREE NUMBERS THAT JUSTIFY LANE 4")
print("=" * 62)
print()
print(f"1. {low_ctr_n:,} of {visible_n:,} visible pages ({low_ctr_pct:.1f}%) have >= 500")
print(f"   impressions but CTR below 0.5%. Large, filterable opportunity pool.")
print()
print(f"2. Median CTR spans a {ratio_str} range across position tiers.")
print(f"   A raw cutoff would be unfair to pages in weaker positions;")
print(f"   position adjustment is necessary — ruling out a simple if-statement.")
print()
print(f"3. Within page-1 results, the IQR of CTR is {p1_iqr_str}.")
print(f"   Position alone doesn't explain CTR; real within-tier spread exists,")
print(f"   giving a scoring model something genuine to learn.")
print()
print("These three numbers confirm: opportunity pool is large, raw CTR is")
print("misleading without tier adjustment, and within-tier signal is real.")

THREE NUMBERS THAT JUSTIFY LANE 4

1. 14,245 of 16,726 visible pages (85.2%) have >= 500
   impressions but CTR below 0.5%. Large, filterable opportunity pool.

2. Median CTR spans a 2.7x (page_1 vs page_3_5 tiers) range across position tiers.
   A raw cutoff would be unfair to pages in weaker positions;
   position adjustment is necessary — ruling out a simple if-statement.

3. Within page-1 results, the IQR of CTR is 0.320% points.
   Position alone doesn't explain CTR; real within-tier spread exists,
   giving a scoring model something genuine to learn.

These three numbers confirm: opportunity pool is large, raw CTR is
misleading without tier adjustment, and within-tier signal is real.


## 4. Careful words: what I can and can't claim

### What this work can say

- **Observational**: "We observed that X% of pages with 500+ impressions and first-page positions have a CTR below the 25th percentile of their position tier in this 90-day snapshot."
- **Directional**: "Pages flagged by the scoring model tended to have lower CTR than their position-tier peers in the held-out client set."
- **Decision-support**: "This ranked list is a suggested starting point for a content editor's audit. It does not guarantee that reviewing a page will improve its CTR."
- **Comparative**: "The position-adjusted score ranked pages better than a raw CTR cutoff at Precision@50 on held-out clients."

### What this work cannot say

| Forbidden claim | Why |
|---|---|
| "Fixing the title caused CTR to increase" | This data has no before/after experiment. Correlation ≠ causation. |
| "These pages violate Google's CTR norms" | We observe deviations from the dataset's own position-tier distribution, not from any Google standard. |
| "The model predicts Google algorithm factors" | We can only observe click patterns in this dataset. We cannot read Google's algorithm. |
| "CTR will recover if the snippet is rewritten" | A recommendation to review — not a guarantee of recovery. |
| "Client X has an engagement problem" | Clients are pseudonymized. We report aggregate patterns only. |

### The leakage watchlist for this lane

- `trend_direction` and `trend_pct` are **never features** (they're the starter label, derived from impression counts in the same window).
- If I define a binary label (e.g., "bottom quartile CTR for its position tier"), the label is computed from `ctr` — so `ctr` itself cannot be a feature. Only its components (`clicks_90d`, `impressions_90d`) or related distinct signals (age, content type, engagement rate) can be.
- `avg_position` defines the tier and may also be a feature — needs care: using position tier (a bucket) as context is fine; using the raw float as a feature while the tier defines the label requires a leakage check.
- The starter dataset is a 90-day snapshot, not a time series. A forward-window label requires the warehouse daily table.

### The honest baseline

Before any model: the simplest possible rule is **"flag any page with 500+ impressions, position 1–20, and CTR below the median for its position tier."** That rule is the **baseline to beat**. A learned model earns its complexity only if it does measurably better at Precision@K on held-out clients — and if it can't, the baseline is the deliverable.

## Self-check

Before submitting, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Picks Lane 4 with a one-paragraph reason
- [x] Names the decision (which pages to audit) and the action (editor reviews title/meta/intent)
- [x] Shows three real numbers from the starter data (pool size, position-tier CTR ratio, within-tier IQR)
- [x] Explains why this is not just 'train a model' (position adjustment must come first; model earns complexity only after baseline)
- [x] Committed to `work/notebooks/` — then submit repo URL on the card